# SparkSession, SparkContext, and Configuration Management

**Module**: 01_spark_fundamentals  
**Notebook**: 03_spark_session_and_spark_context.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How SparkSession acts as the unified gateway, how it encapsulates SparkContext, SQLContext, and HiveContext, and how to tune runtime configurations.
- **Why does this topic exist?**: In modern Spark, SparkSession is your primary interface. Mastering builder patterns, catalogs, and dynamic configuration is required for production engineering.
- **What problem does it solve?**: Prior to Spark 2.0, developers juggled multiple contexts (SQLContext, HiveContext, StreamingContext, SparkContext) causing confusion.
- **Where is it used in real-world Data Engineering?**: Setting up multi-tenant applications, isolated session views, and dynamic partition overwrite parameters.

---

## 2. SparkSession Architecture

```
+---------------------------------------------------------------+
|                        SparkSession                           |
|  - Unified entry point (Spark 2.0+)                           |
|  - Builder pattern: SparkSession.builder.getOrCreate()        |
+-------------------------------+-------------------------------+
                                |
        +-----------------------+-----------------------+
        |                       |                       |
        v                       v                       v
+---------------+       +---------------+       +---------------+
|  SparkContext |       |  Catalog API  |       | Conf Manager  |
|  - Low-level  |       |  - Databases  |       | - spark.conf  |
|  - RDD engine |       |  - Tables     |       | - Runtime     |
|  - Hardware   |       |  - Functions  |       |   parameters  |
+---------------+       +---------------+       +---------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("SparkSession_DeepDive")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Exploring the Catalog API

The Catalog allows programmatically inspecting registered databases, tables, views, and built-in functions.

In [ ]:
# Create a temporary view from sales.csv
sales_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/sales.csv")
sales_df.createOrReplaceTempView("sales_view")

# Query the Catalog
tables = spark.catalog.listTables()
print("Registered Tables and Views in Catalog:")
for t in tables:
    print(f"  Name: {t.name:20} IsTemporary: {t.isTemporary}")

# List available built-in mathematical/string functions in Catalog
funcs = spark.catalog.listFunctions()
print(f"\nTotal built-in functions registered in Catalog: {len(funcs)}")
print("Sample functions:", [f.name for f in funcs[:8]])

## 4. Dynamic Runtime Configuration via `spark.conf`

You can inspect and modify runtime configurations dynamically without restarting the cluster.

In [ ]:
# Read current shuffle partitions
current_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Current shuffle partitions: {current_shuffle}")

# Dynamically change shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "16")
print(f"Updated shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

# Reset back to lab default
spark.conf.set("spark.sql.shuffle.partitions", "8")

---

## Experiment: Isolated Cloned Sessions with `newSession()`

- **1. What are we testing?**: Demonstrate that spark.newSession() creates an isolated session sharing the SparkContext but with independent SQL views and configs.
- **2. What do we expect?**: Temporary views in session A are NOT visible in session B unless registered globally.

In [ ]:
# Session A creates a local temp view
sales_df.createOrReplaceTempView("view_session_a")

# Create Session B
session_b = spark.newSession()

print("Checking 'view_session_a' from Session A:")
print("  Found in Session A:", spark.catalog.tableExists("view_session_a"))

print("Checking 'view_session_a' from Session B:")
print("  Found in Session B:", session_b.catalog.tableExists("view_session_a"))

**3. What happened?**: view_session_a existed in Session A, but tableExists returned False in Session B.  
**4. Why did it happen?**: Temporary views are scoped to individual SparkSession instances.  
**5. Practical Engineering Lesson**: Use newSession() for multi-tenant applications or to prevent temporary view naming collisions.

---

## Practical Exercises

### Exercise 1 (Beginner): Check if a Table Exists in Catalog
**Problem Statement**: Write code using spark.catalog.tableExists() to check if'orders_test'exists.  
**Input / Context**: spark.catalog

In [ ]:
# Exercise 1: Check table existence

### Exercise 2 (Intermediate): Drop a Temporary View Safely
**Problem Statement**: Register a temp view'dummy_view', verify it exists, then drop it using spark.catalog.dropTempView().  
**Input / Context**: spark.catalog.dropTempView

In [ ]:
# Exercise 2: Drop temporary view

### Exercise 3 (Advanced): Global Temporary Views Across Sessions
**Problem Statement**: Register a Global Temporary View and verify that it IS accessible from session_b.  
**Input / Context**: createGlobalTempView('global_sales')

In [ ]:
# Exercise 3: Global Temp Views

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check if a Table Exists in Catalog):
```python
exists = spark.catalog.tableExists("orders_test")
print("Does'orders_test'exist?:", exists)
```

#### Solution 2 (Drop a Temporary View Safely):
```python
spark.range(5).createOrReplaceTempView("dummy_view")
print("Exists before drop:", spark.catalog.tableExists("dummy_view"))
spark.catalog.dropTempView("dummy_view")
print("Exists after drop:", spark.catalog.tableExists("dummy_view"))
```

#### Solution 3 (Global Temporary Views Across Sessions):
```python
sales_df.createOrReplaceGlobalTempView("global_sales")
session_b = spark.newSession()
print("Can Session B query global_temp.global_sales?:")
session_b.sql("SELECT COUNT(*) FROM global_temp.global_sales").show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **SparkSession Purpose**: Unified interface managing Catalog, SQL, and underlying SparkContext.
- **Dynamic Configs**: spark.conf allows inspecting and overriding runtime SQL parameters.
- **Session Scoping**: Temporary views are private to a SparkSession; Global views reside in global_temp database.

### Essential Technical Terms
- `SparkSession`: Unified client entry point.
- `Catalog`: Internal metastore interface for tables, databases, and functions.
- `newSession()`: Creates an isolated session sharing the cluster connection.

### Common Engineering Mistakes
- **Creating Multiple SparkContexts**: A JVM process can only host ONE active SparkContext at a time.
- **Assuming Temp Views are Global**: Temp views disappear across different sessions unless made global.

### Interview Questions on This Topic
- **Q: What is the difference between SparkSession and SparkContext?**
  - *A*: SparkContext is the low-level cluster connection for RDDs; SparkSession wraps SparkContext and adds SQL, DataFrames, and Catalogs.
- **Q: How do you share views across multiple SparkSessions?**
  - *A*: Use createOrReplaceGlobalTempView() and query via global_temp.view_name.

### What to Study Next
Open 01_spark_fundamentals/04_spark_cluster_execution.ipynb to trace execution from action to tasks.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')